In [ ]:
# 1. Cài đặt môi trường
!pip uninstall -y transformers accelerate peft -q
!pip install -q transformers accelerate datasets scikit-learn

import os
import torch
import time
import numpy as np
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

# ==========================================
# CẤU HÌNH
# ==========================================
MODEL_NAME = "bert-base-uncased"
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/MFND_Baseline_BERT"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Tải dữ liệu và Tokenize
print("⏳ Loading dataset...")
dataset = load_dataset("anson-huang/mirage-news")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("🔤 Tokenizing text...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 3. Khởi tạo mô hình
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

# 4. Hàm tính chỉ số (Quan trọng: Phải trả về đúng tên Key)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Trainer trả về logits, ta cần chuyển sang xác suất để tính AUC
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    auc = roc_auc_score(labels, probs[:, 1])

    # Các key này sẽ được Trainer thêm tiền tố 'eval_' hoặc 'test_' tự động
    return {
        "accuracy": acc,
        "f1": f1,
        "auc": auc
    }

# 5. Cấu hình Huấn luyện
training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 6. Huấn luyện
print("🚀 Starting BERT Fine-tuning...")
trainer.train()

# ==========================================
# 7. ĐÁNH GIÁ CHI TIẾT (Xử lý KeyError)
# ==========================================
test_splits = ["test1_nyt_mj", "test2_bbc_dalle", "test3_cnn_dalle", "test4_bbc_sdxl", "test5_cnn_sdxl"]
results_storage = {}

print("\n🔍 ĐÁNH GIÁ TRÊN CÁC TẬP TEST:")
for split in test_splits:
    # predict() thực thi compute_metrics trên tập dữ liệu được truyền vào
    out = trainer.predict(tokenized_datasets[split])

    # Trainer sẽ tự động thêm tiền tố 'test_' vào các key từ compute_metrics
    acc = out.metrics.get("test_accuracy", 0)
    f1 = out.metrics.get("test_f1", 0)
    auc = out.metrics.get("test_auc", 0)

    results_storage[split] = {"acc": acc, "f1": f1, "auc": auc}
    print(f"✅ {split}: Acc={acc*100:.2f}% | F1={f1*100:.2f}% | AUC={auc:.4f}")

# 8. Tổng kết
in_domain = results_storage["test1_nyt_mj"]
ood_vals = [results_storage[s] for s in test_splits[1:]]

print(f"\n📍 In-domain  | Acc: {in_domain['acc']*100:.2f}% | F1: {in_domain['f1']*100:.2f}% | AUC: {in_domain['auc']:.4f}")
print(f"🌐 Avg Out-domain | Acc: {np.mean([x['acc'] for x in ood_vals])*100:.2f}% | F1: {np.mean([x['f1'] for x in ood_vals])*100:.2f}% | AUC: {np.mean([x['auc'] for x in ood_vals]):.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Loading dataset...
🔤 Tokenizing text...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

...

✅ test1_nyt_mj: Acc=94.00% | F1=94.05% | AUC=0.9822


✅ test2_bbc_dalle: Acc=78.20% | F1=81.62% | AUC=0.9285


✅ test3_cnn_dalle: Acc=86.40% | F1=87.02% | AUC=0.9481


✅ test4_bbc_sdxl: Acc=77.40% | F1=81.20% | AUC=0.9393


✅ test5_cnn_sdxl: Acc=86.00% | F1=86.69% | AUC=0.9517

📍 In-domain  | Acc: 94.00% | F1: 94.05% | AUC: 0.9822
🌐 Avg Out-domain | Acc: 82.00% | F1: 84.13% | AUC: 0.9419
